In [ ]:
import pickle
import gzip
from os.path import join

import scanpy.api as sc

import numpy as np
import scipy.sparse as sp
import pandas as pd
from fbpca import pca
import matplotlib.pyplot as plt

%matplotlib inline

In [ ]:
%time adata = sc.read_10x_h5(join("data", "1M_neurons_filtered_gene_bc_matrices_h5.h5"))

In [ ]:
%time sc.pp.recipe_zheng17(adata, plot=True)

In [ ]:
adata.X

In [5]:
x_log = adata.X.copy()
x_log -= x_log.mean(axis=0)
x_log /= x_log.std(axis=0)

In [ ]:
%time U, S, V = pca(x_log, k=50)

Change the sign of the eigenvector so the figures are reproducible.

In [7]:
U[:, np.sum(V, axis=1) < 0] *= -1

In [ ]:
%time x_reduced = np.dot(U, np.diag(S))

In [9]:
x_reduced = x_reduced[:, np.argsort(S)[::-1]][:, :50]

In [ ]:
x_reduced.shape

In [ ]:
cluster_ids = pd.read_csv("data/louvain.csv.gz", header=None, index_col=0, squeeze=True)
cluster_ids.head()

In [12]:
cell_ids = pd.Series(adata.obs.index)

In [13]:
assert all(cluster_ids.index.values.astype(str) == cell_ids.values.astype(str))

In [14]:
data_dict = {"pca_50": x_reduced,
             "CellID": cell_ids.values.astype(str),
             "CellType1": cluster_ids.values}

In [15]:
with gzip.open("data/10x_mouse_zheng.pkl.gz", "wb") as f:
    pickle.dump(data_dict, f)